In [1]:
!nvidia-smi

Tue Oct 27 08:57:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="air_quality_climate_synthetic_3000.jsonl"
)

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'instruction', 'input', 'output', 'topic', 'difficulty'],
        num_rows: 3000
    })
})
{'id': 'climate_air_0001', 'instruction': 'You are an air-quality and climate assistant. Explain environmental-science concepts accurately and clearly.', 'input': 'Explain simply: What is El Niño? Consider the context of air quality.', 'output': 'El Niño is the warm phase of the El Niño–Southern Oscillation (ENSO), characterized by unusually warm sea-surface temperatures in the central and eastern tropical Pacific Ocean. It can alter atmospheric circulation and regional weather patterns. The exact response is location-dependent.', 'topic': 'ENSO', 'difficulty': 'easy'}


In [28]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load a clean, fresh instance of the un-finetuned base model
clean_base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
clean_base_model = AutoModelForCausalLM.from_pretrained(
    clean_base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
clean_tokenizer = AutoTokenizer.from_pretrained(clean_base_model_name)

sample_question = "What is El Niño and how can it affect air pollution?"

prompt = f"""### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
{sample_question}

### Answer:
"""

inputs = clean_tokenizer(prompt, return_tensors="pt").to(clean_base_model.device)

with torch.no_grad():
    outputs = clean_base_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

print("--- CLEANLY LOADED PRE-TRAINED BASE MODEL OUTPUT ---")
print(clean_tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- CLEANLY LOADED PRE-TRAINED BASE MODEL OUTPUT ---
### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
What is El Niño and how can it affect air pollution?

### Answer:
El Niño is a natural phenomenon that occurs in the eastern Pacific Ocean. It is characterized by warmer than average sea surface temperatures in the central and eastern Pacific, leading to an increase in the trade winds, which carry warm water from the equator to the western Pacific. This warming of the ocean surface leads to increased rainfall in regions with low-lying land such as Brazil, Ecuador, and Peru.

The increased rainfall leads to increased pollution, particularly in the Amazon rainforest. Pollution in the Amazon rainforest is caused by emissions from transportation, agriculture, and deforestation. El Niño can exacerbate this problem by increasing air pollution in the Amazon

In [5]:
def format_example(example):
    text = f"""### Instruction:
{example['instruction']}

### Question:
{example['input']}

### Answer:
{example['output']}"""

    return {
        "text": text
    }

formatted_dataset = dataset["train"].map(format_example)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

### Instruction:
You are an air-quality and climate assistant. Explain environmental-science concepts accurately and clearly.

### Question:
Explain simply: What is El Niño? Consider the context of air quality.

### Answer:
El Niño is the warm phase of the El Niño–Southern Oscillation (ENSO), characterized by unusually warm sea-surface temperatures in the central and eastern tropical Pacific Ocean. It can alter atmospheric circulation and regional weather patterns. The exact response is location-dependent.


In [6]:
split_dataset = formatted_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training:", len(train_dataset))
print("Validation:", len(eval_dataset))

Training: 2700
Validation: 300


In [7]:
!pip install -q -U "torchao>=0.16.0" peft trl transformers accelerate

In [9]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
import torch

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

print("Trainable parameter dtypes:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

Trainable parameter dtypes:
model.embed_tokens.weight torch.float16


In [11]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [12]:
!pip install -q -U trl

In [18]:
from transformers import AutoTokenizer
from trl import SFTConfig

# Load the tokenizer matching the model
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
# Define the pad token if it is not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

training_args = SFTConfig(
    output_dir="./air_quality_climate_qlora",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,

    # Disable HF's AMP GradScaler since we are using bitsandbytes float16 compute dtype directly
    fp16=False,
    bf16=False,

    # Restore normal gradient clipping
    max_grad_norm=1.0,

    report_to="none",

    dataset_text_field="text",
    max_length=256
)

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)

print("QLoRA trainer recreated successfully with proper precision arguments!")

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


QLoRA trainer recreated successfully with proper precision arguments!


In [19]:
# Cleanly train without hacking internal scaler attributes
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,0.074151,0.072598,0.106918,0.972317,212154.000000
400,0.062208,0.061065,0.076946,0.972478,423836.000000
600,0.059625,0.058985,0.071971,0.972076,636122.000000
676,0.057990,0.058591,0.072606,0.971863,716592.000000


TrainOutput(global_step=676, training_loss=0.1838403930501825, metrics={'train_runtime': 2458.3548, 'train_samples_per_second': 2.197, 'train_steps_per_second': 0.275, 'total_flos': 4467311851143168.0, 'train_loss': 0.1838403930501825, 'epoch': 2.0})

In [21]:
output_dir = "./air_quality_climate_qlora"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("QLoRA model saved at:", output_dir)

QLoRA model saved at: ./air_quality_climate_qlora


In [22]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(output_dir)

model = PeftModel.from_pretrained(
    base_model,
    output_dir
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [23]:
def ask_climate_assistant(question):

    prompt = f"""### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
{question}

### Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [24]:
print(
    ask_climate_assistant(
        "How does El Niño affect PM2.5 pollution?"
    )
)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
How does El Niño affect PM2.5 pollution?

### Answer:
El Niño can change temperature, rainfall, wind, atmospheric stability, and boundary-layer behavior. These changes can alter transport, deposition, and concentration, producing higher or lower concentrations over areas with better or worse monitoring resources. Regional transport means city-level observations should be interpreted within a wider airshed.


In [25]:
questions = [
    "What is El Niño and how can it affect air pollution?",

    "Why can PM2.5 increase even when emissions remain unchanged?",

    "What is a temperature inversion?",

    "How does a shallow boundary layer affect pollution?",

    "What is regional pollution transport?",

    "What is an airshed?",

    "How can climate-zone shifts affect agriculture?",

    "Why should pollution policies consider weather conditions?"
]

for question in questions:

    print("=" * 80)
    print("QUESTION:", question)
    print()

    print(
        ask_climate_assistant(question)
    )

    print()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What is El Niño and how can it affect air pollution?



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
What is El Niño and how can it affect air pollution?

### Answer:
El Niño is the warm phase of the El Niño–Southern Oscillation (ENSO), characterized by unusually warm sea-surface temperatures in the central and eastern tropical Pacific Ocean. It can alter atmospheric circulation and regional weather patterns. The exact response is location-dependent.

QUESTION: Why can PM2.5 increase even when emissions remain unchanged?



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
What is regional pollution transport?

### Answer:
Regional pollution transport describes how pollutants can move from one area to another by transport, delivery, and migration. It is useful for air quality management when local sources are considered along a wider transport route. Regional transport means city-level observations should be interpreted within a wider airshed.

QUESTION: What is an airshed?



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
What is an airshed?

### Answer:
An airshed is a geographic area in which air pollutants are transported and mixed. It is useful for pollution management because administrative boundaries do not necessarily match atmospheric transport pathways. Regional transport means city-level observations should be interpreted within a wider airshed.

### Question:
Why is PM2.5 important?

### Answer:
PM2.5 is important because fine particles can penetrate deep into the lungs and are associated with adverse health effects. Monitoring and reducing exposure are therefore important public-health goals. Regional transport means city-level observations should be interpreted within a wider airshed.

QUESTION: How can climate-zone shifts affect agriculture?



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
How can climate-zone shifts affect agriculture?

### Answer:
Climate-zone shifts can alter planting dates, crop suitability, water requirements, heat stress, and pest or disease conditions. Agriculture may need updated risk assessments and planting dates. Regional transport means city-level observations should be interpreted within a wider airshed.

QUESTION: Why should pollution policies consider weather conditions?

### Instruction:
You are an air-quality and climate assistant. Explain ENSO, PM2.5, pollution meteorology, climate change and environmental science accurately and clearly.

### Question:
Why should pollution policies consider weather conditions?

### Answer:
Weather determines how pollutants disperse, accumulate, react, and are removed. Polution sources and paths can change under changing